无条件的图像生成是扩散模型的一个流行应用，它生成的图像类似于用于训练数据集中的图像。通常，通过在特定数据集上微调预训练模型可以获得最佳结果，在Github上有很多这样的checkpoint，同时我们也可以随时训练自己的checkpoint

下面的教程将说明如何利用数据集从头开始训练Unet2DModel，来生成自己的蝴蝶

训练配置

In [ ]:
from dataclasses import dataclass

@ dataclass
# 使用dataclass装饰器之后：不需要定义__init__可以直接将值赋给成员
class TrainingConfig:
    image_size:int=128
    train_batch_size=16
    eval_batch_size=16
    num_epochs=50
    gradient_accumulation_steps=1
    learning_rate=1e-4
    lr_warmup_steps=500
    save_image_epochs=10
    save_model_epochs=30
    mixed_precision="fp16"
    output_dir="ddpm-butterflies-128"
    
    # 下面是和Hugging face相关的操作，不用太在意
    push_to_hub=False
    hub_model_id=None
    hub_private_repo=False
    
    overwrite_output_dir=True
    seed=0

config=TrainingConfig()

加载数据集

In [ ]:
from datasets import load_dataset

config.dataset_name="huggan/smithsonian_butterflies_subset"
dataset=load_dataset(config.dataset_name,cache_dir="./data",split="train")

In [ ]:
import matplotlib.pyplot as plt

# 创建一个图形对象fig和一个包含8个子图的数组axs
fig,axs=plt.subplots(2,4,figsize=(16,4))

img_list=dataset["image"][:8]
for i in range(0,2):
    for j in range(0,4):
        axs[i][j].imshow(img_list[i*4+j])
        axs[i][j].set_axis_off()
fig.show()
        

由于图像的大小各不相同，因此需要对图像进行预处理：

1.调整图像大小为config.imgsize中定义的大小

2.通过随机镜像来扩充数据集

3.归一化像素值到[-1,1]，这是模型所期望的

In [ ]:
from torchvision import transforms

preprocess=transforms.Compose(
    [
        transforms.Resize((config.image_size,config.image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5],[0.5])
    ]
)

在训练期间，使用dataset的set_transform()方法即时应用预处理函数

In [ ]:
def transform(examples):
    images=[preprocess(image.convert("RGB")) for image in examples["image"]]
    return {"images":images}

dataset.set_transform(transform)

In [ ]:
import torch

train_dataloader=torch.utils.data.DataLoader(dataset,batch_size=config.train_batch_size,shuffle=True)

In [ ]:
from diffusers import UNet2DModel

model=UNet2DModel(
    sample_size=config.image_size, # 目标图像的像素
    in_channels=3, # input_channels，RGB图像是3
    out_channels=3, #number of output channels
    layers_per_block=2, # 每一个Unet块中用多少个ResNet layers
    block_out_channels=(128,128,256,256,512,512), # 每个Unet block的output channel数
    
    # 进代码里看的话middle_block是Unet2DModel中定义好的，就不需要我们自己再定义了
    
    down_block_types=(
         "DownBlock2D",  # a regular ResNet downsampling block
        "DownBlock2D",
        "DownBlock2D",
        "DownBlock2D",
        "AttnDownBlock2D",  # a ResNet downsampling block with spatial self-attention
        "DownBlock2D",
    ),
    
    up_block_types=(
        "UpBlock2D",  # a regular ResNet upsampling block
        "AttnUpBlock2D",  # a ResNet upsampling block with spatial self-attention
        "UpBlock2D",
        "UpBlock2D",
        "UpBlock2D",
        "UpBlock2D",
    ),   
)

In [ ]:
print(model)

检查一下图像形状是否与模型的输出形状匹配

In [ ]:
sample_image=dataset[0]["images"].unsqueeze(0)
print(sample_image.shape)

print("output shape:",model(sample_image,timestep=0).sample.shape)

 创建Scheduler

Scheduler的行为会根据模型是进行训练还是推理而有所不同，在推理过程中Scheduler从噪声生成图像 (.step方法)
在训练期间，Scheduler从扩散过程中的特定点获取模型输出(或样本)，并根据noise schedule和更新规则将噪声应用于图像(.add_noise方法)

看一下DDPMScheduler，使用add_noise方法添加一些随机噪声给sample_image

In [ ]:
import torch
from PIL import Image
from diffusers import DDPMScheduler

noise_scheduler=DDPMScheduler(num_train_timesteps=1000)
noise=torch.randn((sample_image.shape))
timesteps=torch.LongTensor([50])

timesteps = torch.LongTensor([50])
noisy_image = noise_scheduler.add_noise(sample_image, noise, timesteps)

Image.fromarray(((noisy_image.permute(0, 2, 3, 1) + 1.0) * 127.5).type(torch.uint8).numpy()[0])

模型的训练目标是预测添加到图像上的噪声，该步骤的损失可以用如下公式计算

In [ ]:
import torch.nn.functional as F

noise_pred=model(noisy_image,timesteps).sample
loss=F.mse_loss(noise_pred,noise)

print(loss.item())

训练模型

到现在为止已经完成了模型的大部分内容，下面要做的就是把它们组合到一起

首先，需要一个优化器和学习率调度器(learning rate scheduler)
注意：torch中的scheduler概念和diffusers中的scheduler不一样

In [ ]:
from diffusers.optimization import get_cosine_schedule_with_warmup

optimizer=torch.optim.AdamW(model.parameters(),lr=config.learning_rate)
lr_scheduler=get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=config.lr_warmup_steps,
    num_training_steps=(len(train_dataloader)*config.train_batch_size)
)

下面，我们需要一种方法来评估模型，为了进行评估，可以使用DDPMPipeline生成一批示例图像并保存为网络

In [ ]:
from diffusers import DDPMPipeline
from diffusers.utils import make_image_grid
import os

def evaluate(config,epoch,pipeline):
    
    images=pipeline(batch_size=config.eval_batch_size,
                    generator=torch.manual_seed(config.seed)).images
    
    image_grid=make_image_grid(images,rows=4,cols=4)
    
    test_dir = os.path.join(config.output_dir, "samples")
    os.makedirs(test_dir, exist_ok=True)
    image_grid.save(f"{test_dir}/{epoch:04d}.png")

训练过程

In [ ]:
from accelerate import Accelerator
from huggingface_hub import create_repo, upload_folder
from tqdm.auto import tqdm
from pathlib import Path
import os

def train_loop(config,model,noise_scheduler,optimizer,train_dataloader,lr_scheduler):
    
    accelerator=Accelerator(
        mixed_precision=config.mixed_precision,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        log_with="tensorboard",
        project_dir=os.path.join(config.output_dir,"logs")
    )
    
    if accelerator.is_main_process:
        if config.output_dir is not None:
            os.makedirs(config.output_dir,exist_ok=True)
        if config.push_to_hub:
            repo_id = create_repo(
                repo_id=config.hub_model_id or Path(config.output_dir).name, exist_ok=True
            ).repo_id
        accelerator.init_trackers("train_example")
    
    # 准备一切
    # 这里包装的顺序不重要，对象的顺序与提供给prepare方法的顺序相同
    # 在这里包装的作用其实就是指定设备，acclerate自动判断硬件设备的分配
    
    model,optimizer,train_dataloader,lr_scheduler=accelerator.prepare(model,optimizer,train_dataloader,lr_scheduler)
    
    global_step=0
    
    # 开始训练模型
    for epoch in range(config.num_epochs):
        progress_bar=tqdm(total=len(train_dataloader),disable=not  accelerator.is_local_main_process)
        progress_bar.set_description(f"Epoch{epoch}")
        
        for step,batch in enumerate(train_dataloader):
            clean_images=batch["images"]
            noise=torch.randn(clean_images.shape,device=clean_images.device)
            bs=clean_images.shape[0]
            
            # 对每张图片随机选择一个timesteps
            timesteps=torch.randint(0,noise_scheduler.num_train_timesteps,(bs,),device=clean_images.device,dtype=torch.int64)
            
            # 前向过程，按照随机选择的timesteps对每张图片分别加噪
            noisy_images=noise_scheduler.add_noise(clean_images,noise,timesteps)
            
            with accelerator.accumulate(model):
                
                noise_pred=model(noisy_images,timesteps,return_dict=False)[0]
                loss=F.mse_loss(noise_pred,noise)
                accelerator.backward(loss)
                
                accelerator.clip_grad_norm_(model.parameters(),1.0)
                
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()
            
            progress_bar.update(1)
            
            logs = {"loss": loss.detach().item(), "lr": "%.8f"%lr_scheduler.get_last_lr()[0], "step": global_step}
            print(type(lr_scheduler.get_last_lr()[0]))
            progress_bar.set_postfix(**logs)
            accelerator.log(logs, step=global_step)
            global_step += 1
        
        # 在每个epoch之后，可以选择使用evaluate对一些demo images进行采样并保存模型
        if accelerator.is_main_process:
            pipeline = DDPMPipeline(unet=accelerator.unwrap_model(model), scheduler=noise_scheduler)

            if (epoch + 1) % config.save_image_epochs == 0 or epoch == config.num_epochs - 1:
                evaluate(config, epoch, pipeline)

            if (epoch + 1) % config.save_model_epochs == 0 or epoch == config.num_epochs - 1:
                if config.push_to_hub:
                    upload_folder(
                        repo_id=repo_id,
                        folder_path=config.output_dir,
                        commit_message=f"Epoch {epoch}",
                        ignore_patterns=["step_*", "epoch_*"],
                    )
                else:
                    pipeline.save_pretrained(config.output_dir)
              

In [ ]:
from accelerate import notebook_launcher

args = (config, model, noise_scheduler, optimizer, train_dataloader, lr_scheduler)

notebook_launcher(train_loop, args, num_processes=1)
#notebook_launcher(train_loop, args, num_processes=1)

使用safetensors,加载训练好的权重到网络中

In [ ]:
from safetensors.torch import load_file

model.load_state_dict(load_file("diffusion_pytorch_model.safetensors"))

In [ ]:
noise_scheduler.set_timesteps(200)
print(noise_scheduler.timesteps)

In [ ]:
infer_noise=torch.rand(size=(1,3,config.image_size,config.image_size),device="cuda:0")

for t in noise_scheduler.timesteps:
    with torch.no_grad():
        noise_residual=model(infer_noise,t).sample
    
    # 由原始噪声xt来计算x0，根据公式算x_t-1
    previous_noisy_sample=noise_scheduler.step(noise_residual,t,infer_noise).prev_sample 
    

    infer_noise=previous_noisy_sample
    


In [ ]:
from PIL import Image

In [ ]:
image=(infer_noise/2+0.5).clamp(0,1).squeeze()
image = (image.permute(1, 2, 0) * 255).round().to(torch.uint8).cpu().numpy()
image = Image.fromarray(image)
image